# Treino do córtex do Apolo (Colab)

Treina o `cortex_v0.py` com o currículo em `textos/curriculo/` usando a GPU do Colab e baixa `cortex.pt` + `config.json`.

**Antes de começar:** menu *Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)*.
Depois, execute as células em ordem. Coloque os dois arquivos baixados em `apolo/cerebro/`.

## 1. Conferir a GPU

In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Sem GPU: ative em Ambiente de execução → Alterar tipo de ambiente de execução.")

## 2. Enviar o `apolo_treino.zip`
Selecione o arquivo `apolo_treino.zip` (está em `~/Apolo/apolo/`).

In [ ]:
from google.colab import files
import zipfile, os

enviado = files.upload()
nome = next(iter(enviado))
os.makedirs("apolo", exist_ok=True)
with zipfile.ZipFile(nome) as z:
    z.extractall("apolo")
%cd apolo
!ls -R | head -20

## 3. Treinar
`pequeno` (~10M parâmetros) é o tamanho indicado para GPU. O currículo é pequeno (~95 mil caracteres), então
acompanhe a *perda val*: se ela voltar a subir enquanto a *perda treino* desce, o modelo está decorando — pare e reduza `PASSOS`.

In [ ]:
TAMANHO = "pequeno"   # mini | pequeno | medio
PASSOS = 2000
!python -u -W ignore cortex_v0.py treinar --dados textos/curriculo --tamanho {TAMANHO} --passos {PASSOS} --avaliar_cada 100

## 4. Testar
O Apolo manda a pergunta seguida de uma quebra de linha; o córtex escreve a continuação.

In [ ]:
from cortex_v0 import carregar, continuar

modelo, tk, dev = carregar()
for pergunta in ["Oi", "Bom dia.", "Qual é o seu nome?", "O que é o Apolo?", "Tudo bem?", "Tchau."]:
    texto = continuar(modelo, tk, dev, pergunta + "\n", 120, temperatura=0.5)
    print(f"> {pergunta}\n{texto.split(chr(10) * 2)[0]}\n")

## 5. Baixar os arquivos
Coloque os dois em `apolo/cerebro/` (substituindo os atuais) e avise o Claude Code.

In [ ]:
from google.colab import files
files.download("cerebro/cortex.pt")
files.download("cerebro/config.json")